# 03 — Anomaly detection

Train Isolation Forest and a reconstruction autoencoder on **healthy** cycles, then score the test fleet.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_preprocessing import SENSOR_COLUMNS, prepare_datasets
from src.anomaly_detection import (
    isolation_forest,
    score_isolation_forest,
    train_autoencoder,
    reconstruction_error,
    anomaly_threshold,
    annotate_anomalies,
    save_autoencoder,
    save_iforest,
)
from src.evaluation import binary_anomaly_report, late_life_label

train, test, _ = prepare_datasets()
healthy = train[train["RUL"] > 40]
X_healthy = healthy[SENSOR_COLUMNS].to_numpy(dtype=np.float32)
X_test = test[SENSOR_COLUMNS].to_numpy(dtype=np.float32)
y_risk = late_life_label(test["RUL"].to_numpy(), horizon=20)

In [ ]:
ifo = isolation_forest(X_healthy, contamination=0.08)
ifo_scores = score_isolation_forest(ifo, X_test)
ifo_thr = float(np.quantile(score_isolation_forest(ifo, X_healthy), 0.95))
ifo_pred = (ifo_scores > ifo_thr).astype(int)
print("Isolation Forest vs late-life risk")
print(binary_anomaly_report(y_risk, ifo_pred))
save_iforest(ifo)

In [ ]:
ae = train_autoencoder(X_healthy, epochs=25)
errors = reconstruction_error(ae, X_test)
thr = anomaly_threshold(reconstruction_error(ae, X_healthy), quantile=0.95)
ae_pred = (errors > thr).astype(int)
print("Autoencoder vs late-life risk")
print(binary_anomaly_report(y_risk, ae_pred), "threshold", thr)
save_autoencoder(ae)

In [ ]:
test_ann = annotate_anomalies(test, errors, thr)
sample_id = test_ann["unit_id"].iloc[0]
sub = test_ann[test_ann["unit_id"] == sample_id]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(sub["cycle"], sub["recon_error"], label="recon error")
ax.axhline(thr, color="red", ls="--", label="threshold")
ax.set_title(f"Unit {sample_id} reconstruction error")
ax.legend()
plt.show()